In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from google.colab import files

uploaded = files.upload()

Saving heart_disease_data.csv to heart_disease_data.csv


In [3]:
df = pd.read_csv("heart_disease_data.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (606, 14)

Columns:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [4]:
# Verify target distribution
print("Target distribution:")
print(df["target"].value_counts())

print("\nTarget percentages:")
print(
    (df["target"].value_counts(normalize=True) * 100)
    .round(2)
)

Target distribution:
target
1    330
0    276
Name: count, dtype: int64

Target percentages:
target
1    54.46
0    45.54
Name: proportion, dtype: float64


In [5]:
# entire fuzzy system of rue=le base testing
import numpy as np

def triangular_membership(x, a, b, c):
    x = np.asarray(x, dtype=float)
    membership = np.zeros_like(x)
    if b != a:
        mask = (x >= a) & (x <= b)
        membership[mask] = (x[mask] - a) / (b - a)
    if c != b:
        mask = (x >= b) & (x <= c)
        membership[mask] = (c - x[mask]) / (c - b)
    membership[x == b] = 1.0
    return membership

def trapezoidal_membership(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    membership = np.zeros_like(x)
    if b != a:
        mask = (x >= a) & (x <= b)
        membership[mask] = (x[mask] - a) / (b - a)
    mask = (x >= b) & (x <= c)
    membership[mask] = 1.0
    if d != c:
        mask = (x >= c) & (x <= d)
        membership[mask] = (d - x[mask]) / (d - c)
    return membership

def fuzzify_patient_v2(age, trestbps, chol, thalach, oldpeak, exang, age_min=29, age_max=77, bp_min=94, bp_max=200, chol_min=126, chol_max=564, hr_min=71, hr_max=202, oldpeak_min=0.0, oldpeak_max=6.2):
    return {
        "age": {
            "young": float(trapezoidal_membership(np.array([age]), age_min, age_min, 35, 45)[0]),
            "middle": float(triangular_membership(np.array([age]), 35, 50, 65)[0]),
            "old": float(trapezoidal_membership(np.array([age]), 55, 65, age_max, age_max)[0])
        },
        "blood_pressure": {
            "normal": float(trapezoidal_membership(np.array([trestbps]), bp_min, bp_min, 120, 135)[0]),
            "high": float(triangular_membership(np.array([trestbps]), 120, 140, 160)[0]),
            "very_high": float(trapezoidal_membership(np.array([trestbps]), 145, 160, bp_max, bp_max)[0])
        },
        "cholesterol": {
            "normal": float(trapezoidal_membership(np.array([chol]), chol_min, chol_min, 200, 230)[0]),
            "high": float(triangular_membership(np.array([chol]), 200, 240, 280)[0]),
            "very_high": float(trapezoidal_membership(np.array([chol]), 260, 280, chol_max, chol_max)[0])
        },
        "heart_rate": {
            "low": float(trapezoidal_membership(np.array([thalach]), hr_min, hr_min, 100, 130)[0]),
            "normal": float(triangular_membership(np.array([thalach]), 110, 150, 180)[0]),
            "high": float(trapezoidal_membership(np.array([thalach]), 160, 180, hr_max, hr_max)[0])
        },
        "oldpeak": {
            "low": float(trapezoidal_membership(np.array([oldpeak]), oldpeak_min, oldpeak_min, 1.0, 1.5)[0]),
            "moderate": float(triangular_membership(np.array([oldpeak]), 1.0, 2.0, 3.0)[0]),
            "high": float(trapezoidal_membership(np.array([oldpeak]), 2.5, 3.0, oldpeak_max, oldpeak_max)[0])
        },
        "exercise_angina": {
            "no": 1.0 if exang == 0 else 0.0,
            "yes": 0.0 if exang == 0 else 1.0
        }
    }

def evaluate_rule(rule, fuzzy_data):
    memberships = [fuzzy_data[feature][category] for feature, category in rule["conditions"]]
    return min(memberships) if memberships else 0.0

def evaluate_all_rules(rules, fuzzy_data):
    results = []
    for rule in rules:
        strength = evaluate_rule(rule, fuzzy_data)
        results.append({
            "name": rule["name"],
            "risk": rule["risk"],
            "strength": strength
        })
    return results

def aggregate_risk(rule_evaluations):
    aggregated = {"low": 0.0, "moderate": 0.0, "high": 0.0}
    for eval_item in rule_evaluations:
        risk = eval_item["risk"]
        strength = eval_item["strength"]
        aggregated[risk] = max(aggregated[risk], strength)
    return aggregated

def defuzzify_risk(aggregated_risk):
    risk_centers = {"low": 20.0, "moderate": 50.0, "high": 80.0}
    numerator = sum(aggregated_risk[risk] * risk_centers[risk] for risk in aggregated_risk)
    denominator = sum(aggregated_risk.values())
    if denominator == 0:
        return 0.0
    return numerator / denominator

def get_risk_category(score):
    if score < 35:
        return "low"
    elif score < 65:
        return "moderate"
    else:
        return "high"

def predict_patient_risk(patient, rules):
    fuzzy_data = fuzzify_patient_v2(
        age=patient["age"],
        trestbps=patient["trestbps"],
        chol=patient["chol"],
        thalach=patient["thalach"],
        oldpeak=patient["oldpeak"],
        exang=patient["exang"]
    )
    evaluations = evaluate_all_rules(rules, fuzzy_data)
    aggregated = aggregate_risk(evaluations)
    score = defuzzify_risk(aggregated)
    category = get_risk_category(score)
    return {
        "score": score,
        "category": category,
        "aggregated": aggregated
    }

final_rules = [
    # HIGH
    {"name": "H1", "conditions": [("heart_rate", "high"), ("age", "young")], "risk": "high"},
    {"name": "H2", "conditions": [("heart_rate", "high"), ("oldpeak", "moderate")], "risk": "high"},
    {"name": "H3", "conditions": [("heart_rate", "high"), ("exercise_angina", "no")], "risk": "high"},
    {"name": "H4", "conditions": [("age", "young"), ("oldpeak", "low"), ("exercise_angina", "no")], "risk": "high"},
    {"name": "H5", "conditions": [("cholesterol", "very_high"), ("heart_rate", "high")], "risk": "high"},

    # MODERATE
    {"name": "M1", "conditions": [("age", "middle"), ("blood_pressure", "high"), ("oldpeak", "moderate")], "risk": "moderate"},
    {"name": "M2", "conditions": [("cholesterol", "high"), ("heart_rate", "normal"), ("oldpeak", "moderate")], "risk": "moderate"},
    {"name": "M3", "conditions": [("age", "middle"), ("oldpeak", "moderate"), ("heart_rate", "normal")], "risk": "moderate"},

    # LOW
    {"name": "L1", "conditions": [("oldpeak", "high"), ("exercise_angina", "yes")], "risk": "low"},
    {"name": "L2", "conditions": [("heart_rate", "low"), ("oldpeak", "high")], "risk": "low"},
    {"name": "L3", "conditions": [("cholesterol", "very_high"), ("exercise_angina", "yes")], "risk": "low"},
    {"name": "L4", "conditions": [("age", "old"), ("exercise_angina", "yes")], "risk": "low"}
]

print("Final fuzzy expert system loaded successfully!")
print("Number of rules:", len(final_rules))

print("Fuzzy expert system loaded successfully!")
print("Number of rules:", len(final_rules))
print("Fuzzy expert system loaded successfully!")
print("Number of rules:", len(final_rules))

Final fuzzy expert system loaded successfully!
Number of rules: 12
Fuzzy expert system loaded successfully!
Number of rules: 12
Fuzzy expert system loaded successfully!
Number of rules: 12


In [6]:
def predict_patient_risk(patient, rules):
    fuzzy_data = fuzzify_patient_v2(
        age=patient["age"],
        trestbps=patient["trestbps"],
        chol=patient["chol"],
        thalach=patient["thalach"],
        oldpeak=patient["oldpeak"],
        exang=patient["exang"]
    )

    # Evaluate rules safely
    evaluations = []
    for rule in rules:
        memberships = []
        for feature, category in rule["conditions"]:
            memberships.append(fuzzy_data[feature][category])
        strength = min(memberships) if memberships else 0.0
        evaluations.append({"name": rule["name"], "risk": rule["risk"], "strength": strength})

    # Aggregate risk
    aggregated = {"low": 0.0, "moderate": 0.0, "high": 0.0}
    for eval_item in evaluations:
        risk = eval_item["risk"]
        strength = eval_item["strength"]
        aggregated[risk] = max(aggregated[risk], strength)

    # Defuzzify risk
    risk_centers = {"low": 20.0, "moderate": 50.0, "high": 80.0}
    numerator = sum(aggregated[risk] * risk_centers[risk] for risk in aggregated)
    denominator = sum(aggregated.values())
    score = numerator / denominator if denominator != 0 else 0.0

    # Determine category
    if score < 35:
        category = "low"
    elif score < 65:
        category = "moderate"
    else:
        category = "high"

    return {
        "risk_score": score,
        "risk_category": category,
        "aggregated_risk": aggregated
    }

In [7]:
test_patient = df.iloc[2]

patient_data = {
    "age": test_patient["age"],
    "trestbps": test_patient["trestbps"],
    "chol": test_patient["chol"],
    "thalach": test_patient["thalach"],
    "oldpeak": test_patient["oldpeak"],
    "exang": test_patient["exang"]
}

result = predict_patient_risk(
    patient_data,
    final_rules
)

print("Patient index:", 2)
print("Actual target:", test_patient["target"])

print("\nPatient input:")
print(patient_data)

print("\nFuzzy Risk Result:")
print("Risk Score:", result["risk_score"])
print("Risk Category:", result["risk_category"])

print("\nAggregated Risk:")
print(result["aggregated_risk"])

Patient index: 2
Actual target: 1.0

Patient input:
{'age': np.float64(44.0), 'trestbps': np.float64(118.0), 'chol': np.float64(242.0), 'thalach': np.float64(149.0), 'oldpeak': np.float64(0.3), 'exang': np.float64(0.0)}

Fuzzy Risk Result:
Risk Score: 80.0
Risk Category: high

Aggregated Risk:
{'low': 0.0, 'moderate': 0.0, 'high': 0.1}


In [8]:
# random patient
test_patient = df.iloc[89]

patient_data = {
    "age": test_patient["age"],
    "trestbps": test_patient["trestbps"],
    "chol": test_patient["chol"],
    "thalach": test_patient["thalach"],
    "oldpeak": test_patient["oldpeak"],
    "exang": test_patient["exang"]
}

result = predict_patient_risk(
    patient_data,
    final_rules
)

print("Patient index:", 89)
print("Actual target:", test_patient["target"])

print("\nPatient input:")
print(patient_data)

print("\nFuzzy Risk Result:")
print("Risk Score:", result["risk_score"])
print("Risk Category:", result["risk_category"])

print("\nAggregated Risk:")
print(result["aggregated_risk"])

Patient index: 89
Actual target: 1.0

Patient input:
{'age': np.float64(64.0), 'trestbps': np.float64(140.0), 'chol': np.float64(313.0), 'thalach': np.float64(133.0), 'oldpeak': np.float64(0.2), 'exang': np.float64(0.0)}

Fuzzy Risk Result:
Risk Score: 0.0
Risk Category: low

Aggregated Risk:
{'low': 0.0, 'moderate': 0.0, 'high': 0.0}


In [9]:
import inspect

print(inspect.getsource(get_risk_category))

def get_risk_category(score):
    if score < 35:
        return "low"
    elif score < 65:
        return "moderate"
    else:
        return "high"



In [10]:
# updated risk classifaction

def get_risk_category(score):

    # No rule produced any evidence
    if score is None:
        return "insufficient_evidence"

    # Risk categories
    if score < 40:
        return "low"
    elif score < 70:
        return "moderate"
    else:
        return "high"


print("Risk classification function updated successfully!")

Risk classification function updated successfully!


In [11]:
# random patient
test_patient = df.iloc[89]

patient_data = {
    "age": test_patient["age"],
    "trestbps": test_patient["trestbps"],
    "chol": test_patient["chol"],
    "thalach": test_patient["thalach"],
    "oldpeak": test_patient["oldpeak"],
    "exang": test_patient["exang"]
}

result = predict_patient_risk(
    patient_data,
    final_rules
)

print("Patient index:", 89)
print("Actual target:", test_patient["target"])

print("\nFuzzy Risk Result:")
print("Risk Score:", result["risk_score"])
print("Risk Category:", result["risk_category"])

print("\nAggregated Risk:")
print(result["aggregated_risk"])

Patient index: 89
Actual target: 1.0

Fuzzy Risk Result:
Risk Score: 0.0
Risk Category: low

Aggregated Risk:
{'low': 0.0, 'moderate': 0.0, 'high': 0.0}


In [12]:
import inspect

print(inspect.getsource(defuzzify_risk))

def defuzzify_risk(aggregated_risk):
    risk_centers = {"low": 20.0, "moderate": 50.0, "high": 80.0}
    numerator = sum(aggregated_risk[risk] * risk_centers[risk] for risk in aggregated_risk)
    denominator = sum(aggregated_risk.values())
    if denominator == 0:
        return 0.0
    return numerator / denominator



In [13]:
def defuzzify_risk(aggregated_risk):

    risk_centers = {
        "low": 20.0,
        "moderate": 50.0,
        "high": 80.0
    }

    numerator = sum(
        aggregated_risk[risk] * risk_centers[risk]
        for risk in aggregated_risk
    )

    denominator = sum(aggregated_risk.values())

    # No rule fired
    if denominator == 0:
        return None

    return numerator / denominator


print("Defuzzification function updated successfully!")

Defuzzification function updated successfully!


In [14]:
def get_risk_category(score):

    if score is None:
        return "insufficient_evidence"

    if score < 40:
        return "low"
    elif score < 70:
        return "moderate"
    else:
        return "high"


print("Risk classification function updated successfully!")

Risk classification function updated successfully!


In [15]:
test_patient = df.iloc[76]

patient_data = {
    "age": test_patient["age"],
    "trestbps": test_patient["trestbps"],
    "chol": test_patient["chol"],
    "thalach": test_patient["thalach"],
    "oldpeak": test_patient["oldpeak"],
    "exang": test_patient["exang"]
}

result = predict_patient_risk(
    patient_data,
    final_rules
)

print("Patient index:", 76)
print("Actual target:", test_patient["target"])

print("\nFuzzy Risk Result:")
print("Risk Score:", result["risk_score"])
print("Risk Category:", result["risk_category"])

print("\nAggregated Risk:")
print(result["aggregated_risk"])

Patient index: 76
Actual target: 1.0

Fuzzy Risk Result:
Risk Score: 80.0
Risk Category: high

Aggregated Risk:
{'low': 0.0, 'moderate': 0.0, 'high': 0.1}


In [16]:
# Run the fuzzy expert system on all 606 patients

evaluation_results = []

for index, row in df.iterrows():

    patient = {
        "age": row["age"],
        "trestbps": row["trestbps"],
        "chol": row["chol"],
        "thalach": row["thalach"],
        "oldpeak": row["oldpeak"],
        "exang": row["exang"]
    }

    result = predict_patient_risk(
        patient,
        final_rules
    )

    evaluation_results.append({
        "index": index,
        "actual_target": row["target"],
        "risk_score": result["risk_score"],
        "risk_category": result["risk_category"]
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Evaluation completed!")
print("Total patients evaluated:", len(evaluation_df))

evaluation_df.head(10)

Evaluation completed!
Total patients evaluated: 606


,index,actual_target,risk_score,risk_category
0,0,0.0,0.000000,low
1,1,0.0,80.000000,high
2,2,1.0,80.000000,high
3,3,0.0,20.000000,low
4,4,1.0,54.285714,moderate
5,5,0.0,0.000000,low
6,6,0.0,50.000000,moderate
7,7,0.0,20.000000,low
8,8,1.0,0.000000,low
9,9,0.0,43.114754,moderate


In [17]:

# Calculate how many patients received an actual fuzzy prediction

total_patients = len(evaluation_df)

# A zero score alone cannot tell us whether a rule fired.
# Check the prediction output for an explicit insufficient-evidence label.
insufficient_count = (
    evaluation_df["risk_category"]
    .astype(str)
    .str.lower()
    .isin(["insufficient_evidence", "insufficient evidence"])
    .sum()
)

covered_count = total_patients - insufficient_count

coverage_percentage = (
    covered_count / total_patients
) * 100

print("TOTAL PATIENTS:", total_patients)
print("PATIENTS WITH A RISK CATEGORY:", covered_count)
print("INSUFFICIENT-EVIDENCE PATIENTS:", insufficient_count)
print(f"RULE COVERAGE: {coverage_percentage:.2f}%")

TOTAL PATIENTS: 606
PATIENTS WITH A RISK CATEGORY: 606
INSUFFICIENT-EVIDENCE PATIENTS: 0
RULE COVERAGE: 100.00%


In [18]:
# check the risk category distribuction

print("Risk category distribution:")
print(
    evaluation_df["risk_category"]
    .value_counts(dropna=False)
)

print("\nRisk category percentages:")
print(
    (
        evaluation_df["risk_category"]
        .value_counts(normalize=True, dropna=False) * 100
    ).round(2)
)

Risk category distribution:
risk_category
low         302
high        200
moderate    104
Name: count, dtype: int64

Risk category percentages:
risk_category
low         49.83
high        33.00
moderate    17.16
Name: proportion, dtype: float64


In [19]:

# Verify coverage using aggregated fuzzy rule strengths

verified_results = []

for index, row in df.iterrows():

    patient = {
        "age": row["age"],
        "trestbps": row["trestbps"],
        "chol": row["chol"],
        "thalach": row["thalach"],
        "oldpeak": row["oldpeak"],
        "exang": row["exang"]
    }

    result = predict_patient_risk(patient, final_rules)

    aggregated = result["aggregated_risk"]
    total_strength = sum(aggregated.values())
    rule_fired = total_strength > 0

    verified_results.append({
        "index": index,
        "actual_target": int(row["target"]),
        "risk_score": result["risk_score"],
        "risk_category": result["risk_category"],
        "rule_fired": rule_fired,
        "total_rule_strength": total_strength
    })

verified_df = pd.DataFrame(verified_results)

total = len(verified_df)
covered = int(verified_df["rule_fired"].sum())
uncovered = total - covered

print("Total patients:", total)
print("Patients with at least one fired rule:", covered)
print("Patients with no fired rules:", uncovered)
print(f"Verified rule coverage: {covered / total * 100:.2f}%")

print("\nExamples with no fired rules:")
display(
    verified_df.loc[
        ~verified_df["rule_fired"]
    ].head(10)
)

Total patients: 606
Patients with at least one fired rule: 446
Patients with no fired rules: 160
Verified rule coverage: 73.60%

Examples with no fired rules:


,index,actual_target,risk_score,risk_category,rule_fired,total_rule_strength
0,0,0,0.0,low,False,0.0
5,5,0,0.0,low,False,0.0
8,8,1,0.0,low,False,0.0
11,11,1,0.0,low,False,0.0
12,12,1,0.0,low,False,0.0
13,13,1,0.0,low,False,0.0
17,17,1,0.0,low,False,0.0
18,18,1,0.0,low,False,0.0
21,21,0,0.0,low,False,0.0
22,22,0,0.0,low,False,0.0


In [20]:

# Create corrected evaluation labels
# Patients with no fired rules are marked as insufficient_evidence.

verified_df["evaluated_category"] = verified_df["risk_category"]

verified_df.loc[
    ~verified_df["rule_fired"],
    "evaluated_category"
] = "insufficient_evidence"

print("Corrected category distribution:")
print(
    verified_df["evaluated_category"]
    .value_counts()
)

print("\nCategory percentages:")
print(
    (
        verified_df["evaluated_category"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

Corrected category distribution:
evaluated_category
high                     200
insufficient_evidence    160
low                      142
moderate                 104
Name: count, dtype: int64

Category percentages:
evaluated_category
high                     33.00
insufficient_evidence    26.40
low                      23.43
moderate                 17.16
Name: proportion, dtype: float64


In [21]:
# confusion martrices and accuracy

from sklearn.metrics import confusion_matrix, accuracy_score
import pandas as pd

# Include only patients for whom at least one rule fired
covered_df = verified_df[
    verified_df["rule_fired"]
].copy()

# Binary evaluation convention:
# Low risk = 0 (predicted no disease)
# Moderate/High risk = 1 (predicted disease)

covered_df["predicted_target"] = (
    covered_df["evaluated_category"]
    .isin(["moderate", "high"])
    .astype(int)
)

y_true = covered_df["actual_target"].astype(int)
y_pred = covered_df["predicted_target"]

# Calculate confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

accuracy = accuracy_score(y_true, y_pred)

print("Total patients in dataset:", len(df))
print("Patients covered by rules:", len(covered_df))
print("Patients with no fired rules:", len(df) - len(covered_df))

print("\nConfusion Matrix:")
print("Rows = Actual [No Disease, Disease]")
print("Columns = Predicted [No Disease, Disease]")
print(cm)

print(f"\nAccuracy on covered patients: {accuracy * 100:.2f}%")
print(f"Rule coverage: {len(covered_df) / len(df) * 100:.2f}%")

Total patients in dataset: 606
Patients covered by rules: 446
Patients with no fired rules: 160

Confusion Matrix:
Rows = Actual [No Disease, Disease]
Columns = Predicted [No Disease, Disease]
[[118  92]
 [ 24 212]]

Accuracy on covered patients: 73.99%
Rule coverage: 73.60%
